# CUPED Variance-Reduced A/B Testing

**Reference**: Deng, Xu, Kohavi, Walker — *Improving the Sensitivity of Online Controlled Experiments by Utilizing Pre-Experiment Data*, WSDM 2013 (Microsoft)

## What CUPED does

Standard A/B test noise comes from user heterogeneity — heavy users always spend more, light users always less. CUPED exploits a pre-experiment covariate X (e.g., last month's revenue) that's correlated with the outcome Y to subtract out predictable per-user variance:

$$Y_{\text{cuped}} = Y - \theta(X - \bar{X}), \quad \theta = \frac{\text{Cov}(Y, X)}{\text{Var}(X)}$$

This is equivalent to a one-covariate OLS regression adjustment. The variance reduction is:

$$\text{Var}(Y_{\text{cuped}}) = \text{Var}(Y)(1 - \rho^2)$$

Since required sample size scales with variance, CUPED needs **$(1 - \rho^2)$ times fewer users** to detect the same effect.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

from data_gen import generate_experiment
from analysis import standard_ttest, cuped_ttest, cuped_adjust, variance_reduction_report
from power import required_sample_size, mde, sample_size_savings

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1  Distribution overlay: raw Y vs Y_cuped

At ρ=0.7, CUPED tightens the per-user metric distribution — less noise for the same signal.

In [ ]:
df = generate_experiment(n_users=20_000, rho=0.7, delta=0.05)
adjusted = cuped_adjust(df)

fig, ax = plt.subplots(figsize=(9, 4))
kw = dict(bins=120, density=True, alpha=0.6)
ax.hist(df['post_metric'], label=f'Raw Y  (std={df["post_metric"].std():.3f})', **kw)
ax.hist(adjusted['y_cuped'], label=f'Y_cuped  (std={adjusted["y_cuped"].std():.3f})', **kw)
ax.set_xlabel('Metric value')
ax.set_ylabel('Density')
ax.set_title('Distribution of raw Y vs. CUPED-adjusted Y  (ρ=0.7)')
ax.legend()
plt.tight_layout()
plt.show()

rpt = variance_reduction_report(df)
print(f"Empirical variance reduction : {rpt['empirical_reduction']*100:.1f}%")
print(f"Theoretical (1-ρ²)           : {rpt['theoretical_reduction']*100:.1f}%")

## 2  Required sample size: Raw vs. CUPED across ρ values

In [ ]:
delta = 0.05
rhos = np.linspace(0, 0.95, 60)

# Use the empirical sigma from a reference dataset (rho=0)
df_ref = generate_experiment(n_users=50_000, rho=0.0, delta=delta)
sigma_raw = df_ref['post_metric'].std()

n_raw_vals = [required_sample_size(sigma_raw, delta)] * len(rhos)
n_cuped_vals = [required_sample_size(sigma_raw * np.sqrt(1 - r**2), delta) for r in rhos]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rhos, n_raw_vals, '--', label='Raw t-test', lw=2)
ax.plot(rhos, n_cuped_vals, label='CUPED', lw=2)
ax.fill_between(rhos, n_cuped_vals, n_raw_vals, alpha=0.15, label='Savings')
ax.set_xlabel('Pre/post correlation ρ')
ax.set_ylabel('Required N per arm')
ax.set_title('Sample-size requirements: Raw vs. CUPED (MDE=0.05)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for r in [0.3, 0.5, 0.7, 0.9]:
    savings = sample_size_savings(r, required_sample_size(sigma_raw, delta))
    print(f"ρ={r}: {savings['reduction_pct']:.0f}% fewer users needed  ({savings['n_raw']:,} → {savings['n_cuped']:,})")

## 3  p-value vs. N curves: how much sooner does CUPED reach significance?

In [ ]:
rho = 0.7
delta = 0.05
n_vals = np.arange(500, 30_001, 500)

n_seeds = 30
raw_pvals = np.zeros((len(n_vals), n_seeds))
cuped_pvals = np.zeros((len(n_vals), n_seeds))

for j, seed in enumerate(range(n_seeds)):
    df_full = generate_experiment(n_users=30_000, rho=rho, delta=delta, seed=seed)
    for i, n in enumerate(n_vals):
        sub = df_full.groupby('variant', group_keys=False).apply(
            lambda g: g.head(n // 2)
        ).reset_index(drop=True)
        raw_pvals[i, j] = standard_ttest(sub).p_value
        cuped_pvals[i, j] = cuped_ttest(sub).p_value

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(n_vals, np.median(raw_pvals, axis=1), '--', label='Raw t-test (median p)', lw=2)
ax.semilogy(n_vals, np.median(cuped_pvals, axis=1), label='CUPED (median p)', lw=2)
ax.axhline(0.05, color='red', lw=1, linestyle=':', label='α=0.05')
ax.set_xlabel('Sample size per arm')
ax.set_ylabel('p-value (log scale)')
ax.set_title(f'p-value vs. N  (ρ={rho}, δ={delta}, median over {n_seeds} seeds)')
ax.legend()
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## 4  Variance reduction vs. correlation sweep

Confirms the theoretical 1−ρ² relationship empirically.

In [ ]:
rhos_sweep = np.arange(0, 0.96, 0.05)
empirical_reductions = []
theoretical_reductions = []

for r in rhos_sweep:
    df_s = generate_experiment(n_users=30_000, rho=r, delta=0.1)
    rpt = variance_reduction_report(df_s)
    empirical_reductions.append(rpt['empirical_reduction'] * 100)
    theoretical_reductions.append(rpt['theoretical_reduction'] * 100)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rhos_sweep, theoretical_reductions, '--', label='Theoretical (1−ρ²)', lw=2)
ax.scatter(rhos_sweep, empirical_reductions, label='Empirical', zorder=5, s=40)
ax.set_xlabel('Pre/post correlation ρ')
ax.set_ylabel('Variance reduction (%)')
ax.set_title('CUPED variance reduction: theoretical vs. empirical')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nSummary at ρ=0.7:")
df70 = generate_experiment(n_users=50_000, rho=0.7, delta=0.05)
raw = standard_ttest(df70)
cup = cuped_ttest(df70)
svgs = sample_size_savings(0.7, required_sample_size(df70['post_metric'].std(), 0.05))
print(f"  Raw estimate:   {raw.estimate:.4f}  p={raw.p_value:.4f}")
print(f"  CUPED estimate: {cup.estimate:.4f}  p={cup.p_value:.4f}")
print(f"  Sample-size savings: {svgs['reduction_pct']:.0f}%  ({svgs['n_raw']:,} → {svgs['n_cuped']:,} users)")